> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAGzTA-js2I/lm1uXy0dKQBcrruDsbJt5w/view?utm_content=DAGzTA-js2I&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=hf88292377f)。


# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

# 2. 链式流程

在 LangChain 中，**链（Chain）** 是指将多个组件（如提示词、模型、解析器等）按照一定的逻辑顺序组合在一起，从而实现端到端的任务处理。下面我们依次介绍主要的链类型和它们的演进。

## 2.1 LLMChain 

**LLMChain** 是 LangChain 早期的核心链，但目前已经被淘汰。

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import LLMChain
from langchain_core.prompts import ChatPromptTemplate
import os

# 1. 定义提示词模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个智能 AI 助手"),
    ("human", "当前问题：{current_question}")
])

# 2. 定义语言模型
llm = ChatOpenAI(
    model="ernie-3.5-8k",
    openai_api_key=os.environ.get("OPENAI_API_KEY"),
    base_url="https://aistudio.baidu.com/llm/lmapi/v3"
)

# 3. 构建 LLMChain
conversation = LLMChain(
    llm=llm,
    prompt=prompt
)

# 简单演示
question = "一句话人工智能的应用有哪些？"
response = conversation.invoke({"current_question": question})
print("模型回答:", response["text"])

## 2.2 LCEL（LangChain Expression Language）

### 2.2.1 LCEL 简介

**LCEL**（LangChain Expression Language）是 LangChain 在 0.2 版本中引入的一种全新构建链的方式。它使用管道符 `|` 将不同的组件灵活串联起来，让链式调用的可读性和扩展性大幅提升，真正实现了模块化、声明式的工作流设计。

### 2.2.2 RunnableLambda 封装函数节点

#### 应用场景1：类型审查

我们可以设置一个简单的 lambda 函数，从而实现传入一个内容 x ，输出打印输入 x 的类型，然后把原样的 x 再传下去 。这样我们把这部分内容接入到合适的位置里，就能看到 Runnable 对象的输入和输出格式了。


In [ ]:
def debug(x):
    print("🧪 当前数据类型：", type(x))
    return x

比如这里我们就可以看到 `prompt` 需要传入的格式是 `<class 'dict'>` 字典。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
import os
prompt = ChatPromptTemplate.from_template("用一句话介绍 {topic}")
llm = ChatOpenAI(model="ernie-3.5-8k",
openai_api_key=os.environ.get("OPENAI_API_KEY"),
base_url="https://aistudio.baidu.com/llm/lmapi/v3")
debug = RunnableLambda(lambda x: print("🧪 当前数据类型：", type(x)) or x)
chain = debug | prompt | llm
print(chain.invoke({"topic":"广州"}).content)

这里我们也就可以观察到 `prompt` 的输出和 `llm` 的输入应该是支持 `<class 'langchain_core.prompt_values.ChatPromptValue'> `。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
import os
prompt = ChatPromptTemplate.from_template("用一句话介绍 {topic}")
llm = ChatOpenAI(model="ernie-3.5-8k",
openai_api_key=os.environ.get("OPENAI_API_KEY"),
base_url="https://aistudio.baidu.com/llm/lmapi/v3")
debug = RunnableLambda(lambda x: print("🧪 当前数据类型：", type(x)) or x)
chain = prompt | debug | llm
print(chain.invoke({"topic":"广州"}).content)

这里我们也就可以观察到 `llm` 的输出应该是 `<class 'langchain_core.messages.ai.AIMessage'>` 。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
import os
prompt = ChatPromptTemplate.from_template("用一句话介绍 {topic}")
llm = ChatOpenAI(model="ernie-3.5-8k",
openai_api_key=os.environ.get("OPENAI_API_KEY"),
base_url="https://aistudio.baidu.com/llm/lmapi/v3")
debug = RunnableLambda(lambda x: print("🧪 当前数据类型：", type(x)) or x)
chain = prompt | llm | debug
print(chain.invoke({"topic":"广州"}).content)

#### 应用场景2：插入函数

实际上 RunnableLambda 不仅仅能封装 Lambda 函数，正常的 python 函数也能够进行封装。比如下面我们就在输入的字典里加了一个表情符🔥。比如下面的例子里我们传入的是一个字典 `{"topic":"开心"}` ，那么当经过这个函数以后就会变成 `{"topic":"开心🔥"}` 。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
import os
prompt = ChatPromptTemplate.from_template("用一句话介绍 {topic}")
llm = ChatOpenAI(model="ernie-3.5-8k",
openai_api_key=os.environ.get("OPENAI_API_KEY"),
base_url="https://aistudio.baidu.com/llm/lmapi/v3")

def add_fire(input):
    input["topic"] += "🔥"
    return input

chain = RunnableLambda(add_fire) | prompt | llm 

print(chain.invoke({"topic":"开心"}).content)

在 LangChain 中，除了使用 RunnableLambda 以外，我们还可以用一个装饰器 @chain 来定义一个函数成为 Runnable 对象。然后我们可以直接使用该函数名称载入到 LCEL 中。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import chain
import os
prompt = ChatPromptTemplate.from_template("用一句话介绍 {topic}")
llm = ChatOpenAI(model="ernie-3.5-8k",
openai_api_key=os.environ.get("OPENAI_API_KEY"),
base_url="https://aistudio.baidu.com/llm/lmapi/v3")

@chain
def add_fire(input):
    input["topic"] += "🔥"
    return input

chain = add_fire | prompt | llm 

print(chain.invoke({"topic":"开心"}).content)

### 2.2.3 使用 RunnableMap 并行处理多个字段

有些任务我们希望同时处理多个数据项，比如多个提示词或多个模型，这时候就可以使用 **RunnableMap** 来实现。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableMap
import os

prompt1 = ChatPromptTemplate.from_template("你是百度制作的大模型，一句话回答用户向你提出问题： {topic}")
prompt2 = ChatPromptTemplate.from_template("你是OpenAI制作的大模型，一句话回答用户向你提出问题： {topic}")
llm = ChatOpenAI(model="ernie-3.5-8k",
openai_api_key=os.environ.get("OPENAI_API_KEY"),
base_url="https://aistudio.baidu.com/llm/lmapi/v3")

chain = RunnableMap({
    "a_result": prompt1 | llm,
    "b_result": prompt2 | llm,
})

print(chain.invoke({"topic":"你是谁？"}))
print(chain.invoke({"topic":"你是谁？"})["a_result"].content)
print(chain.invoke({"topic":"你是谁？"})["b_result"].content)

### 2.2.4 LCEL 总结

- 通过上面的介绍，我们可以看到，LCEL 的出现为 LangChain 带来了全新的工作流编排方式。它不仅让 Prompt → LLM → 解析 → 后处理 的链路表达更加直观，还通过 Runnable 接口、管道式语法 和 可组合组件，让我们可以随时接入记忆、检索、并行处理等复杂功能，极大提升了应用的灵活性和可扩展性。

- 相比早期的 LLMChain，LCEL 更像是一种 声明式工作流语言，它统一了输入输出规范，支持异步与流式执行，同时与 LangSmith 等工具深度集成，方便我们进行可视化调试与性能优化。未来随着 LangChain 生态的不断丰富，LCEL 也将成为构建智能体、多模型协同应用的核心基石。

## 2.3 组合链（SequentialChain）

在 LangChain 中，除了最基础的 **LLMChain（或现在推荐的 LCEL 链）** 之外，还可以把多个链组合在一起，形成一个更复杂的工作流，这就是**组合链（Sequential Chains）**。

### 2.3.1 SimpleSequentialChain

在这个示例中，我们的目标是：通过输入一个主题，最终获取与该主题相关的关键词。但如果直接让大模型一步到位地生成关键词，往往不够精准。因此我们采用**分步骤拆解**的方式。

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
import os

llm = ChatOpenAI(model="ernie-3.5-8k",
openai_api_key=os.environ.get("OPENAI_API_KEY"),
base_url="https://aistudio.baidu.com/llm/lmapi/v3")

prompt_1 = PromptTemplate.from_template("写一段关于{input}的简短文章。")
chain_1 = prompt_1 | llm

prompt_2 = PromptTemplate.from_template("请总结以下文章：{input}")
chain_2 = prompt_2 | llm

prompt_3 = PromptTemplate.from_template("请提取以下内容的关键词：{input}")
chain_3 = prompt_3 | llm

map_input = RunnableLambda(lambda x: {"input": x.content})

# 将三个子链串起来
overall_chain = (
    chain_1 |
    map_input | chain_2 |
    map_input | chain_3
)

# 调用整个链
result = overall_chain.invoke({"input": "人工智能教育"})
print(result.content)


当然这里我们也可以用 @chain 来进行实现：

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
from langchain_core.runnables import chain

llm = ChatOpenAI(model="ernie-3.5-8k",
openai_api_key=os.environ.get("OPENAI_API_KEY"),
base_url="https://aistudio.baidu.com/llm/lmapi/v3")

prompt_1 = PromptTemplate.from_template("写一段关于{input}的简短文章。")
chain_1 = prompt_1 | llm

prompt_2 = PromptTemplate.from_template("请总结以下文章：{input}")
chain_2 = prompt_2 | llm

prompt_3 = PromptTemplate.from_template("请提取以下内容的关键词：{input}")
chain_3 = prompt_3 | llm

map_input = RunnableLambda(lambda x: {"input": x.content})

# 将三个子链串起来
@chain
def overall_chain(input):
  result1 = chain_1.invoke({"input": input}).content
  result2 = chain_2.invoke({"input": result1}).content
  result3 = chain_3.invoke({"input": result2})
  return result3

# 调用整个链
result = overall_chain.invoke({"input": "人工智能教育"})
print(result.content)

### 2.3.2 Sequential Chain

在 SimpleSequentialChain 的基础上，SequentialChain 是一个更加灵活和复杂的链式结构。其允许多个链同时处理不同的输入，并且可以将多个输出合并或分别传递到后续的链中。

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
from langchain_core.runnables import RunnableMap, RunnableLambda

# 初始化模型
llm = ChatOpenAI(model="ernie-3.5-8k",
openai_api_key=os.environ.get("OPENAI_API_KEY"),
base_url="https://aistudio.baidu.com/llm/lmapi/v3")

# ===== Step 1: 翻译评论 =====
first_prompt = PromptTemplate.from_template("将以下评论翻译成中文：{Review}")
chain_one = first_prompt | llm  # LCEL 里用 | 代替 LLMChain

# ===== Step 2: 提取关键问题 =====
second_prompt = PromptTemplate.from_template("请提取评论中最关键的问题：{Chinese_Review}")
chain_two = second_prompt | llm

# ===== Step 3: 识别语言 =====
third_prompt = PromptTemplate.from_template("以下评论是用什么语言写的：{Review}")
chain_three = third_prompt | llm

# ===== Step 4: 生成后续回复 =====
fourth_prompt = PromptTemplate.from_template(
    "根据以下总结和指定语言，以商家的视角，写一条对应语言的后续回复（最终只需要输出一条）：\n\n总结: {summary}\n\n语言: {language}"
)
chain_four = fourth_prompt | llm

# ===== 组合逻辑：先串行，再传递必要变量 =====
overall_chain = (
    RunnableMap({
        "summary": chain_one | RunnableLambda(lambda x: {"Chinese_Review": x.content}) | chain_two,  # 提取问题
        "language": chain_three                      # 识别语言
    })
    | RunnableLambda(lambda x: {
        "summary": x["summary"].content,
        "language": x["language"].content
    })
    | chain_four                                    # 生成后续回复
)

# 运行
result = overall_chain.invoke({"Review": "我好喜欢这个产品呀！"})
print(result.content)

这里我们也可以用 @chain 来进行改造：

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
from langchain_core.runnables import chain

llm = ChatOpenAI(model="ernie-3.5-8k",
openai_api_key=os.environ.get("OPENAI_API_KEY"),
base_url="https://aistudio.baidu.com/llm/lmapi/v3")

# ===== Step 1: 翻译评论 =====
first_prompt = PromptTemplate.from_template("将以下评论翻译成中文：{Review}")
chain_one = first_prompt | llm  # LCEL 里用 | 代替 LLMChain

# ===== Step 2: 提取关键问题 =====
second_prompt = PromptTemplate.from_template("请提取评论中最关键的问题：{Chinese_Review}")
chain_two = second_prompt | llm

# ===== Step 3: 识别语言 =====
third_prompt = PromptTemplate.from_template("以下评论是用什么语言写的：{Review}")
chain_three = third_prompt | llm

# ===== Step 4: 生成后续回复 =====
fourth_prompt = PromptTemplate.from_template(
    "根据以下总结和指定语言，以商家的视角，写一条对应语言的后续回复（最终只需要输出一条）：\n\n总结: {summary}\n\n语言: {language}"
)
chain_four = fourth_prompt | llm

# ===== 组合逻辑：先串行，再传递必要变量 =====
@chain
def overall_chain(input):
  chinese_review = chain_one.invoke({"Review": input["Review"]}).content
  summary = chain_two.invoke({"Chinese_Review": chinese_review}).content
  language = chain_three.invoke({"Review": input["Review"]}).content
  result = chain_four.invoke({"summary": summary, "language": language})
  return result

# 运行
result = overall_chain.invoke({"Review": "我好喜欢这个产品呀！"})
print(result.content)